# Load data

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_json(
    '../data/raw/shopee_reviews_dataset.jsonl',
    lines=True
)
df.head()

In [ ]:
print(df.shape)
print(df.info())

In [ ]:
df_unaccented = pd.read_json(
    '../data/raw/aug_unaccented_reviews.jsonl',
    lines=True
)
df_unaccented.head()

In [ ]:
print(df_unaccented.shape)
print(df_unaccented.info())

In [ ]:
print("NA counts:\n", df.isna().sum())
print("NA counts:\n", df_unaccented.isna().sum())

# Clean text

In [ ]:
import re
import string

# syntax: re.sub(pattern, replace, text)
# find pattern and replace with 'replace'
# r"..." is raw string
# \S means character that  is not whitespace
# .*? means all characters
def remove_url(text):
    return re.sub(r"http\S+|www\S+", "", text)

def remove_html(text):
    return re.sub(r"<.*?>", "", text)

def remove_punctuation(text):
    return re.sub(r'[^\w\s]', ' ', text)
    # delete all punctuations

def remove_special_characters(text):
    return re.sub(r"[^a-zA-ZÀ-ỹ0-9\s]", "", text)

def normalize_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()
    # turn consecutive whitespaces into one, remove begin/end whitespace

def clean_text(text):
    text = text.lower()
    text = remove_url(text)
    text = remove_html(text)
    text = remove_punctuation(text)
    text = remove_special_characters(text)
    text = normalize_whitespace(text)

    return text

In [ ]:
df["clean_review"] = df["review"].apply(clean_text)
display(df.loc[0, "review"])
display(df["clean_review"])


In [ ]:
df_unaccented["clean_review"] = df_unaccented["review"].apply(clean_text)
display(df_unaccented.loc[0,"review"])
display(df_unaccented["clean_review"])

# Normalize_text

From this step forward, the unaccented reviews CAN'T be processed the same way with accented ones. It is really hard and may be out of scope to try to give diatritic to those unaccented reviews.

In [ ]:
df.loc[1, "review"]

In [ ]:
df.loc[3405, "review"]

In [ ]:
import unicodedata

SLANG_DICT = {
    "ko": "không",
    "khum": "không",
    "hok": "không",
    "k": "không",
    "dc": "được",
    "mn": "mọi người",
    "ntn": "như thế nào",
    "sp": "sản phẩm"
}
def normalize_unicode(text):
    return unicodedata.normalize("NFC", text)

def normalize_slang(text):
    words = text.split()
    norm_words = []
    for word in words:
        norm_words.append(SLANG_DICT.get(word, word))
        # replace if it's a slang
    return " ".join(norm_words)

def normalize_repeated_characters(text):
    return re.sub(r"(.)\1+", r"\1", text)

def normalize_text(text):
    text = normalize_unicode(text)
    text = normalize_slang(text)
    text = normalize_repeated_characters(text)

    return text

In [ ]:
df["normalized_review"] = df["clean_review"].apply(normalize_text)
display(df.loc[1:5, "clean_review"])
display(df["normalized_review"])

# Word_segmentation

In this step, we define what is a correct word to preserve the meaning of words. This step is critical because Vietnamese word is not word by word separated by spaces, but a combination, for example "hoc sinh" not "hoc" and "sinh". We have 2 libraries to help with this.

In [ ]:
from underthesea import word_tokenize
from pyvi.ViTokenizer import tokenize

def segment_underthesea(text):
    return word_tokenize(text, format="text")
    # segment and return a string (format="text")

def segment_pyvi(text):
    return tokenize(text)

def segment_text(text, method="underthesea"):
    if method == "underthesea":
        return segment_underthesea(text)
    elif method == "pyvi":
        return segment_pyvi(text)
    else:
        raise ValueError("Invalid segmentation method")

In [ ]:
df["segmented_review"] = df["normalized_review"].apply(segment_pyvi)
display(df.loc[1:5, "normalized_review"])
display(df["segmented_review"])

# Tokenize

TF-IDF is used for ML. PhoBERT is used for DL like LSTM. This is where ML and DL pipeline sets apart. Tokenization returns a sparse matrix or tensor, not normal vector or dataframe type.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoTokenizer

def tokenize_tfidf(tokenizer, texts):
    return tokenizer.fit_transform(texts)

def phobert_tokenizer():
    return AutoTokenizer.from_pretrained("vinai/phobert-base")

def tokenize_phobert(tokenizer, text):
    return tokenizer(text, padding="max_length", # force all sequences to have the same length
                     truncation=True, # truncate if token too long
                     max_length=128, return_tensor="pt") # return pytorch tensor

In [ ]:
tfidf = TfidfVectorizer()
tokens_ML = tokenize_tfidf(tfidf, df["segmented_review"])
tokens_ML

In [ ]:
phobert = phobert_tokenizer()
tokens_DL = df["segmented_review"].apply(
    lambda x: tokenize_phobert(phobert, x)
)
tokens_DL